In [13]:
import pandas as pd
import numpy as np
import gzip
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [14]:
metadata_path = "CRyPTIC_reuse_table_20240917.csv"

df = pd.read_csv(metadata_path)

print("Total samples:", len(df))
df.head()

Total samples: 12287


,ENA_RUN,UNIQUEID,AMI_BINARY_PHENOTYPE,BDQ_BINARY_PHENOTYPE,CFZ_BINARY_PHENOTYPE,DLM_BINARY_PHENOTYPE,EMB_BINARY_PHENOTYPE,ETH_BINARY_PHENOTYPE,INH_BINARY_PHENOTYPE,KAN_BINARY_PHENOTYPE,...,INH_PHENOTYPE_QUALITY,KAN_PHENOTYPE_QUALITY,LEV_PHENOTYPE_QUALITY,LZD_PHENOTYPE_QUALITY,MXF_PHENOTYPE_QUALITY,RIF_PHENOTYPE_QUALITY,RFB_PHENOTYPE_QUALITY,ENA_SAMPLE,VCF,REGENOTYPED_VCF
0,ERR4810489,site.02.subj.0001.lab.2014222001.iso.1,S,NaN,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298516,../reproducibility/00/01/08/61/10861/site.02.i...,../reproducibility/00/01/08/61/10861/site.02.i...
1,ERR4810491,site.02.subj.0002.lab.2014222005.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,LOW,HIGH,HIGH,ERS5298518,../reproducibility/00/01/08/63/10863/site.02.i...,../reproducibility/00/01/08/63/10863/site.02.i...
2,ERR4810493,site.02.subj.0004.lab.2014222010.iso.1,S,S,S,NaN,S,I,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298520,../reproducibility/00/01/08/67/10867/site.02.i...,../reproducibility/00/01/08/67/10867/site.02.i...
3,ERR4810494,site.02.subj.0005.lab.2014222011.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298521,../reproducibility/00/01/08/68/10868/site.02.i...,../reproducibility/00/01/08/68/10868/site.02.i...
4,ERR4810495,site.02.subj.0006.lab.2014222013.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298522,../reproducibility/00/01/08/69/10869/site.02.i...,../reproducibility/00/01/08/69/10869/site.02.i...


In [15]:
def convert_label(x):
    
    if x == "R":
        return 1
    
    elif x == "S":
        return 0
    
    else:
        return np.nan

In [16]:
drug_columns = [
    "RIF_BINARY_PHENOTYPE",
    "INH_BINARY_PHENOTYPE",
    "EMB_BINARY_PHENOTYPE"
]

for col in drug_columns:
    df[col] = df[col].apply(convert_label)

df = df.dropna(subset=drug_columns)

print("Samples after filtering:", len(df))

Samples after filtering: 10289


In [18]:
VCF_DIR = "vcf2024"
def extract_mutations(vcf_path):

    mutations = set()

    with gzip.open(vcf_path, "rt") as f:
        
        for line in f:
            
            if line.startswith("#"):
                continue
            
            parts = line.strip().split("\t")
            
            pos = parts[1]
            ref = parts[3]
            alt = parts[4]
            
            mut = f"{pos}_{ref}_{alt}"
            
            mutations.add(mut)

    return mutations

In [20]:
all_mutations = set()
sample_mutations = {}

for idx, row in tqdm(df.iterrows(), total=len(df)):

    sample = row["ENA_SAMPLE"]   # use ENA_SAMPLE instead of ENA_RUN

    vcf_path = os.path.join("vcf(2024)", f"{sample}.vcf.gz")

    if not os.path.exists(vcf_path):
        continue

    muts = extract_mutations(vcf_path)

    sample_mutations[sample] = muts

    all_mutations.update(muts)

print("Total unique mutations:", len(all_mutations))
print("Samples processed:", len(sample_mutations))

100%|██████████| 10289/10289 [00:13<00:00, 745.14it/s]

Total unique mutations: 1086959
Samples processed: 10289


In [21]:
mutation_list = sorted(list(all_mutations))

print("Total mutation features:", len(mutation_list))

Total mutation features: 1086959


In [23]:
mutation_counts = {}

for muts in sample_mutations.values():
    
    for m in muts:
        
        mutation_counts[m] = mutation_counts.get(m, 0) + 1

print("Total mutations before filtering:", len(mutation_counts))

Total mutations before filtering: 1086959


In [24]:
MIN_COUNT = 5

filtered_mutations = {
    m for m, count in mutation_counts.items() if count >= MIN_COUNT
}

print("Mutations after filtering:", len(filtered_mutations))

Mutations after filtering: 86932


In [25]:
mutation_list = sorted(filtered_mutations)

print("Final feature count:", len(mutation_list))

Final feature count: 86932


In [26]:
matrix = []
samples = []

for sample, muts in tqdm(sample_mutations.items()):
    
    row = [1 if m in muts else 0 for m in mutation_list]
    
    matrix.append(row)
    samples.append(sample)

X = pd.DataFrame(matrix, columns=mutation_list, index=samples)

print("Matrix shape:", X.shape)

100%|██████████| 10289/10289 [01:25<00:00, 119.90it/s]


Matrix shape: (10289, 86932)


In [27]:
labels = df.set_index("ENA_SAMPLE")[drug_columns]

labels = labels.loc[X.index]

labels.head()

,RIF_BINARY_PHENOTYPE,INH_BINARY_PHENOTYPE,EMB_BINARY_PHENOTYPE
ERS5298516,0.0,0.0,0.0
ERS5298518,0.0,0.0,0.0
ERS5298520,0.0,0.0,0.0
ERS5298521,0.0,0.0,0.0
ERS5298522,0.0,0.0,0.0


In [29]:
# Split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    labels,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (7202, 86932)
Validation: (1543, 86932)
Test: (1544, 86932)


In [38]:
#save train, val, test sets
X_train.to_csv("X_train.csv")
y_train.to_csv("y_train.csv")
X_val.to_csv("X_val.csv")
y_val.to_csv("y_val.csv")
X_test.to_csv("X_test.csv")
y_test.to_csv("y_test.csv")

In [34]:
import torch
import torch.nn as nn
from dataclasses import dataclass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

@dataclass(frozen=True)
class WDNNConfig:
    input_dim: int = 86392
    hidden_dim: int = 1000
    output_dim: int = 3
    dropout_rate: float = 0.3

    # Matches the extracted SavedModel settings
    bn_epsilon: float = 1e-3
    bn_momentum_keras: float = 0.99

    # Regularizer coefficients from extracted architecture
    kernel_l1: float = 1e-4
    hidden_bias_l2: float = 1e-3
    output_bias_l2: float = 1e-1

class ExactWDNN(nn.Module):
    def __init__(self, config: WDNNConfig) -> None:
        super().__init__()
        self.config = config

        # PyTorch BN momentum uses update weight; Keras uses decay.
        bn_momentum_torch = 1.0 - self.config.bn_momentum_keras

        self.dense1 = nn.Linear(self.config.input_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout = nn.Dropout(p=self.config.dropout_rate)

        self.dense2 = nn.Linear(self.config.hidden_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization_1 = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout_1 = nn.Dropout(p=self.config.dropout_rate)

        self.dense3 = nn.Linear(self.config.hidden_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization_2 = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout_2 = nn.Dropout(p=self.config.dropout_rate)

        self.wdnn_final_layer = nn.Linear(self.config.hidden_dim, self.config.output_dim, bias=True)
        self.output_activation = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dense1(x)
        x = torch.relu(x)
        x = self.batch_normalization(x)
        x = self.dropout(x)

        x = self.dense2(x)
        x = torch.relu(x)
        x = self.batch_normalization_1(x)
        x = self.dropout_1(x)

        x = self.dense3(x)
        x = torch.relu(x)
        x = self.batch_normalization_2(x)
        x = self.dropout_2(x)

        x = self.wdnn_final_layer(x)
        x = self.output_activation(x)
        return x

    def regularization_loss(self) -> torch.Tensor:
        loss = torch.zeros((), dtype=self.dense1.weight.dtype, device=self.dense1.weight.device)

        dense_layers = [self.dense1, self.dense2, self.dense3]
        for layer in dense_layers:
            loss = loss + self.config.kernel_l1 * layer.weight.abs().sum()
            loss = loss + self.config.hidden_bias_l2 * layer.bias.pow(2).sum()

        loss = loss + self.config.kernel_l1 * self.wdnn_final_layer.weight.abs().sum()
        loss = loss + self.config.output_bias_l2 * self.wdnn_final_layer.bias.pow(2).sum()
        return loss

# Build model from current notebook data dimensions
config = WDNNConfig(
    input_dim=X_train.shape[1],
    output_dim=y_train.shape[1],
)

model = ExactWDNN(config).to(device)
print(model)

Using device: cuda
ExactWDNN(
  (dense1): Linear(in_features=86932, out_features=1000, bias=True)
  (batch_normalization): BatchNorm1d(1000, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (dense2): Linear(in_features=1000, out_features=1000, bias=True)
  (batch_normalization_1): BatchNorm1d(1000, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (dropout_1): Dropout(p=0.3, inplace=False)
  (dense3): Linear(in_features=1000, out_features=1000, bias=True)
  (batch_normalization_2): BatchNorm1d(1000, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (dropout_2): Dropout(p=0.3, inplace=False)
  (wdnn_final_layer): Linear(in_features=1000, out_features=3, bias=True)
  (output_activation): Sigmoid()
)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import copy
import os

# 10-fold multilabel stratified shuffle split settings
N_FOLDS = 10
VAL_SIZE = 0.20
RANDOM_STATE = 42
EPOCHS = 20
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
THRESHOLD = 0.5

SAVE_FOLD_CHECKPOINTS = True
CV_CHECKPOINT_DIR = "checkpoints_cv"
if SAVE_FOLD_CHECKPOINTS:
    os.makedirs(CV_CHECKPOINT_DIR, exist_ok=True)

X_np = X.values.astype(np.float32)
y_np = labels.values.astype(np.float32)
y_np_int = y_np.astype(np.int64)

def safe_div(num, den):
    return float(num / den) if den > 0 else np.nan

def compute_multilabel_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(np.int64)
    per_drug = []

    for i, drug in enumerate(drug_columns):
        yt = y_true[:, i].astype(np.int64)
        yp = y_pred[:, i].astype(np.int64)
        yp_prob = y_prob[:, i].astype(np.float64)

        tp = int(((yp == 1) & (yt == 1)).sum())
        tn = int(((yp == 0) & (yt == 0)).sum())
        fp = int(((yp == 1) & (yt == 0)).sum())
        fn = int(((yp == 0) & (yt == 1)).sum())

        sensitivity = safe_div(tp, tp + fn)
        specificity = safe_div(tn, tn + fp)
        precision = safe_div(tp, tp + fp)
        npv = safe_div(tn, tn + fn)

        try:
            auc = roc_auc_score(yt, yp_prob)
        except ValueError:
            auc = np.nan

        per_drug.append({
            "drug": drug,
            "AUC": auc,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "NPV": npv,
        })

    per_drug_df = pd.DataFrame(per_drug)
    macro_metrics = {
        "AUC": float(np.nanmean(per_drug_df["AUC"])),
        "sensitivity": float(np.nanmean(per_drug_df["sensitivity"])),
        "specificity": float(np.nanmean(per_drug_df["specificity"])),
        "precision": float(np.nanmean(per_drug_df["precision"])),
        "NPV": float(np.nanmean(per_drug_df["NPV"])),
    }
    return per_drug_df, macro_metrics

msss = MultilabelStratifiedShuffleSplit(
    n_splits=N_FOLDS,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    )

fold_macro_rows = []
fold_per_drug_rows = []

for fold, (train_idx, val_idx) in enumerate(msss.split(X_np, y_np_int), start=1):
    print(f"\n===== Fold {fold}/{N_FOLDS} =====")

    X_train_fold = torch.tensor(X_np[train_idx], dtype=torch.float32)
    y_train_fold = torch.tensor(y_np[train_idx], dtype=torch.float32)
    X_val_fold = torch.tensor(X_np[val_idx], dtype=torch.float32)
    y_val_fold = torch.tensor(y_np[val_idx], dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train_fold, y_train_fold),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_fold, y_val_fold),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    fold_config = WDNNConfig(
        input_dim=X_np.shape[1],
        output_dim=y_np.shape[1],
    )
    model = ExactWDNN(fold_config).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_loss = float("inf")
    best_epoch = 0
    best_state_dict = None
    best_macro = None
    best_per_drug = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb) + model.regularization_loss()
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * xb.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)

        model.eval()
        val_loss_sum = 0.0
        val_probs_list = []
        val_targets_list = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb) + model.regularization_loss()
                val_loss_sum += loss.item() * xb.size(0)
                val_probs_list.append(preds.detach().cpu().numpy())
                val_targets_list.append(yb.detach().cpu().numpy())

        val_loss = val_loss_sum / len(val_loader.dataset)
        val_probs = np.vstack(val_probs_list)
        val_targets = np.vstack(val_targets_list)

        per_drug_df, macro = compute_multilabel_metrics(
            y_true=val_targets,
            y_prob=val_probs,
            threshold=THRESHOLD,
        )

        print(
            f"Fold {fold} Epoch {epoch}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | "
            f"AUC: {macro['AUC']:.4f} | Sens: {macro['sensitivity']:.4f} | "
            f"Spec: {macro['specificity']:.4f} | Prec: {macro['precision']:.4f} | NPV: {macro['NPV']:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            best_macro = macro
            best_per_drug = per_drug_df.copy()

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    if SAVE_FOLD_CHECKPOINTS and best_state_dict is not None:
        fold_ckpt_path = os.path.join(CV_CHECKPOINT_DIR, f"wdnn_fold_{fold:02d}_best.pt")
        torch.save(
            {
                "fold": fold,
                "best_epoch": best_epoch,
                "model_state_dict": best_state_dict,
                "config": fold_config,
                "best_val_loss": best_val_loss,
                "best_macro_metrics": best_macro,
            },
            fold_ckpt_path,
        )
        print(f"Saved fold {fold} best checkpoint: {fold_ckpt_path}")

    fold_row = {
        "fold": fold,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "AUC": best_macro["AUC"],
        "sensitivity": best_macro["sensitivity"],
        "specificity": best_macro["specificity"],
        "precision": best_macro["precision"],
        "NPV": best_macro["NPV"],
    }
    fold_macro_rows.append(fold_row)

    best_per_drug = best_per_drug.copy()
    best_per_drug.insert(0, "fold", fold)
    best_per_drug.insert(1, "best_epoch", best_epoch)
    fold_per_drug_rows.append(best_per_drug)

fold_macro_df = pd.DataFrame(fold_macro_rows)
fold_per_drug_df = pd.concat(fold_per_drug_rows, ignore_index=True)

print("\n===== 10-Fold Macro Summary (best epoch per fold) =====")
display(fold_macro_df)

summary_mean = fold_macro_df[["AUC", "sensitivity", "specificity", "precision", "NPV"]].mean()
summary_std = fold_macro_df[["AUC", "sensitivity", "specificity", "precision", "NPV"]].std()
summary_df = pd.DataFrame({"mean": summary_mean, "std": summary_std})

print("\n===== Overall Mean +/- Std Across Folds =====")
display(summary_df)

print("\n===== Per-Drug Metrics Across Folds (best epoch per fold) =====")
display(fold_per_drug_df)


===== Fold 1/10 =====
Fold 1 Epoch 1/20 | Train loss: 11.8286 | Val loss: 8.8127 | AUC: 0.7450 | Sens: 0.9797 | Spec: 0.0677 | Prec: 0.3336 | NPV: 0.8787
Fold 1 Epoch 2/20 | Train loss: 6.7088 | Val loss: 5.4553 | AUC: 0.7643 | Sens: 0.8003 | Spec: 0.4088 | Prec: 0.4047 | NPV: 0.8705
Fold 1 Epoch 3/20 | Train loss: 4.6218 | Val loss: 4.2979 | AUC: 0.7287 | Sens: 0.7070 | Spec: 0.5372 | Prec: 0.4192 | NPV: 0.7961
Fold 1 Epoch 4/20 | Train loss: 3.8524 | Val loss: 3.6735 | AUC: 0.7828 | Sens: 0.4934 | Spec: 0.8618 | Prec: 0.6846 | NPV: 0.7767
Fold 1 Epoch 5/20 | Train loss: 3.5625 | Val loss: 3.4222 | AUC: 0.9101 | Sens: 0.6137 | Spec: 0.9268 | Prec: 0.8227 | NPV: 0.8437
Fold 1 Epoch 6/20 | Train loss: 3.1777 | Val loss: 3.0759 | AUC: 0.9090 | Sens: 0.7389 | Spec: 0.8739 | Prec: 0.7751 | NPV: 0.8923
Fold 1 Epoch 7/20 | Train loss: 2.9080 | Val loss: 2.9182 | AUC: 0.8942 | Sens: 0.3795 | Spec: 0.9679 | Prec: 0.8748 | NPV: 0.7776
Fold 1 Epoch 8/20 | Train loss: 2.7617 | Val loss: 2.8957 |

,fold,best_epoch,best_val_loss,AUC,sensitivity,specificity,precision,NPV
0,1,20,2.313971,0.929208,0.561018,0.819406,0.809404,0.855110
1,2,20,2.149102,0.946855,0.735356,0.941541,0.868469,0.887102
2,3,20,2.077448,0.932420,0.610455,0.974687,0.906987,0.842404
3,4,19,2.165723,0.917029,0.506740,0.941174,0.861231,0.824142
4,5,17,2.224462,0.931741,0.707620,0.937661,0.883706,0.878696
5,6,20,2.176839,0.925650,0.535587,0.971307,0.916803,0.831483
6,7,20,2.160563,0.949176,0.710169,0.960960,0.900239,0.896327
7,8,20,2.357307,0.938256,0.194328,0.995925,0.947849,0.731171
8,9,16,2.225171,0.943016,0.460464,0.987015,0.948749,0.809420
9,10,20,2.187304,0.936032,0.841556,0.883521,0.785394,0.925551



===== Overall Mean +/- Std Across Folds =====


,mean,std
AUC,0.934938,0.009880
sensitivity,0.586329,0.181670
specificity,0.941320,0.053422
precision,0.882883,0.053926
NPV,0.848141,0.054652



===== Per-Drug Metrics Across Folds (best epoch per fold) =====


,fold,best_epoch,drug,AUC,sensitivity,specificity,precision,NPV
0,1,20,RIF_BINARY_PHENOTYPE,0.934402,0.440476,0.985570,0.936709,0.784156
1,1,20,INH_BINARY_PHENOTYPE,0.934370,0.965792,0.480102,0.579740,0.949749
2,1,20,EMB_BINARY_PHENOTYPE,0.918851,0.276786,0.992547,0.911765,0.831426
3,2,20,RIF_BINARY_PHENOTYPE,0.941882,0.802083,0.932900,0.852848,0.906732
4,2,20,INH_BINARY_PHENOTYPE,0.940034,0.787913,0.915326,0.873578,0.853197
5,2,20,EMB_BINARY_PHENOTYPE,0.958649,0.616071,0.976398,0.878981,0.901376
6,3,20,RIF_BINARY_PHENOTYPE,0.928824,0.583333,0.974026,0.915888,0.828221
7,3,20,INH_BINARY_PHENOTYPE,0.928914,0.696693,0.977985,0.959184,0.812808
8,3,20,EMB_BINARY_PHENOTYPE,0.939521,0.551339,0.972050,0.845890,0.886183
9,4,19,RIF_BINARY_PHENOTYPE,0.920947,0.605655,0.971140,0.910515,0.835506


In [3]:
# Evaluate best saved fold on test set without rerunning previous cells
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

drug_columns = [
    "RIF_BINARY_PHENOTYPE",
    "INH_BINARY_PHENOTYPE",
    "EMB_BINARY_PHENOTYPE",
]
THRESHOLD = 0.5
CV_CHECKPOINT_DIR = "checkpoints_cv"
X_TEST_PATH = "X_test.csv"
Y_TEST_PATH = "y_test.csv"

@dataclass(frozen=True)
class WDNNConfig:
    input_dim: int = 86392
    hidden_dim: int = 1000
    output_dim: int = 3
    dropout_rate: float = 0.3
    bn_epsilon: float = 1e-3
    bn_momentum_keras: float = 0.99
    kernel_l1: float = 1e-4
    hidden_bias_l2: float = 1e-3
    output_bias_l2: float = 1e-1

class ExactWDNN(nn.Module):
    def __init__(self, config: WDNNConfig) -> None:
        super().__init__()
        self.config = config
        bn_momentum_torch = 1.0 - self.config.bn_momentum_keras

        self.dense1 = nn.Linear(self.config.input_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout = nn.Dropout(p=self.config.dropout_rate)

        self.dense2 = nn.Linear(self.config.hidden_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization_1 = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout_1 = nn.Dropout(p=self.config.dropout_rate)

        self.dense3 = nn.Linear(self.config.hidden_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization_2 = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout_2 = nn.Dropout(p=self.config.dropout_rate)

        self.wdnn_final_layer = nn.Linear(self.config.hidden_dim, self.config.output_dim, bias=True)
        self.output_activation = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dense1(x)
        x = torch.relu(x)
        x = self.batch_normalization(x)
        x = self.dropout(x)

        x = self.dense2(x)
        x = torch.relu(x)
        x = self.batch_normalization_1(x)
        x = self.dropout_1(x)

        x = self.dense3(x)
        x = torch.relu(x)
        x = self.batch_normalization_2(x)
        x = self.dropout_2(x)

        x = self.wdnn_final_layer(x)
        x = self.output_activation(x)
        return x

def safe_div(num, den):
    return float(num / den) if den > 0 else np.nan

def compute_multilabel_metrics(y_true, y_prob, drug_cols, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(np.int64)
    per_drug = []

    for i, drug in enumerate(drug_cols):
        yt = y_true[:, i].astype(np.int64)
        yp = y_pred[:, i].astype(np.int64)
        yp_prob = y_prob[:, i].astype(np.float64)

        tp = int(((yp == 1) & (yt == 1)).sum())
        tn = int(((yp == 0) & (yt == 0)).sum())
        fp = int(((yp == 1) & (yt == 0)).sum())
        fn = int(((yp == 0) & (yt == 1)).sum())

        sensitivity = safe_div(tp, tp + fn)
        specificity = safe_div(tn, tn + fp)
        precision = safe_div(tp, tp + fp)
        npv = safe_div(tn, tn + fn)

        try:
            auc = roc_auc_score(yt, yp_prob)
        except ValueError:
            auc = np.nan

        per_drug.append({
            "drug": drug,
            "AUC": auc,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "NPV": npv,
        })

    per_drug_df = pd.DataFrame(per_drug)
    macro_metrics = {
        "AUC": float(np.nanmean(per_drug_df["AUC"])),
        "sensitivity": float(np.nanmean(per_drug_df["sensitivity"])),
        "specificity": float(np.nanmean(per_drug_df["specificity"])),
        "precision": float(np.nanmean(per_drug_df["precision"])),
        "NPV": float(np.nanmean(per_drug_df["NPV"])),
    }
    return per_drug_df, macro_metrics

def load_checkpoint(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

if not os.path.isdir(CV_CHECKPOINT_DIR):
    raise FileNotFoundError(f"Checkpoint directory not found: {CV_CHECKPOINT_DIR}")
if not os.path.exists(X_TEST_PATH):
    raise FileNotFoundError(f"Missing test features file: {X_TEST_PATH}")
if not os.path.exists(Y_TEST_PATH):
    raise FileNotFoundError(f"Missing test labels file: {Y_TEST_PATH}")

checkpoint_rows = []
for fname in sorted(os.listdir(CV_CHECKPOINT_DIR)):
    if not (fname.startswith("wdnn_fold_") and fname.endswith("_best.pt")):
        continue
    ckpt_path = os.path.join(CV_CHECKPOINT_DIR, fname)
    ckpt = load_checkpoint(ckpt_path, map_location="cpu")
    fold_value = ckpt.get("fold")
    if fold_value is None:
        continue
    macro = ckpt.get("best_macro_metrics", {}) or {}
    checkpoint_rows.append({
        "fold": int(fold_value),
        "path": ckpt_path,
        "AUC": float(macro.get("AUC", np.nan)),
        "sensitivity": float(macro.get("sensitivity", np.nan)),
        "specificity": float(macro.get("specificity", np.nan)),
        "precision": float(macro.get("precision", np.nan)),
        "NPV": float(macro.get("NPV", np.nan)),
    })

if not checkpoint_rows:
    raise RuntimeError("No fold checkpoints found in checkpoints_cv.")

fold_macro_df = pd.DataFrame(checkpoint_rows).sort_values(
    by=["AUC", "fold"], ascending=[False, True], na_position="last"
).reset_index(drop=True)

best_fold = int(fold_macro_df.loc[0, "fold"])
best_fold_ckpt_path = fold_macro_df.loc[0, "path"]
print(f"\nBest fold based on saved checkpoint AUC: Fold {best_fold}")
display(fold_macro_df[["fold", "AUC", "sensitivity", "specificity", "precision", "NPV"]])

X_test = pd.read_csv(X_TEST_PATH, index_col=0)
y_test = pd.read_csv(Y_TEST_PATH, index_col=0)

missing_cols = [c for c in drug_columns if c not in y_test.columns]
if missing_cols:
    raise ValueError(f"Missing drug columns in y_test: {missing_cols}")

common_idx = X_test.index.intersection(y_test.index)
if len(common_idx) == 0:
    raise RuntimeError("No overlapping sample IDs between X_test and y_test.")

X_test = X_test.loc[common_idx]
y_test = y_test.loc[common_idx, drug_columns]

checkpoint = load_checkpoint(best_fold_ckpt_path, map_location=device)
cfg_obj = checkpoint.get("config")

cfg_values = {}
if isinstance(cfg_obj, dict):
    for key in WDNNConfig.__dataclass_fields__.keys():
        if key in cfg_obj:
            cfg_values[key] = cfg_obj[key]
elif cfg_obj is not None:
    for key in WDNNConfig.__dataclass_fields__.keys():
        if hasattr(cfg_obj, key):
            cfg_values[key] = getattr(cfg_obj, key)

cfg_values["input_dim"] = int(X_test.shape[1])
cfg_values["output_dim"] = len(drug_columns)

base_cfg = WDNNConfig()
best_fold_config = WDNNConfig(
    input_dim=int(cfg_values.get("input_dim", base_cfg.input_dim)),
    hidden_dim=int(cfg_values.get("hidden_dim", base_cfg.hidden_dim)),
    output_dim=int(cfg_values.get("output_dim", base_cfg.output_dim)),
    dropout_rate=float(cfg_values.get("dropout_rate", base_cfg.dropout_rate)),
    bn_epsilon=float(cfg_values.get("bn_epsilon", base_cfg.bn_epsilon)),
    bn_momentum_keras=float(cfg_values.get("bn_momentum_keras", base_cfg.bn_momentum_keras)),
    kernel_l1=float(cfg_values.get("kernel_l1", base_cfg.kernel_l1)),
    hidden_bias_l2=float(cfg_values.get("hidden_bias_l2", base_cfg.hidden_bias_l2)),
    output_bias_l2=float(cfg_values.get("output_bias_l2", base_cfg.output_bias_l2)),
)

best_model = ExactWDNN(best_fold_config).to(device)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

X_test_tensor = torch.tensor(X_test.values.astype(np.float32), dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values.astype(np.float32), dtype=torch.float32).to(device)

with torch.no_grad():
    test_preds = best_model(X_test_tensor).cpu().numpy()
    test_targets = y_test_tensor.cpu().numpy()

test_per_drug_df, test_macro = compute_multilabel_metrics(
    y_true=test_targets,
    y_prob=test_preds,
    drug_cols=drug_columns,
    threshold=THRESHOLD,
)

print(f"\n===== Test Set Metrics for Best Fold (Fold {best_fold}) =====")
print(
    f"AUC: {test_macro['AUC']:.4f} | Sensitivity: {test_macro['sensitivity']:.4f} | "
    f"Specificity: {test_macro['specificity']:.4f} | Precision: {test_macro['precision']:.4f} | NPV: {test_macro['NPV']:.4f}"
)
print("\nPer-drug metrics on test set:")
display(test_per_drug_df)

Using device: cuda

Best fold based on saved checkpoint AUC: Fold 7


,fold,AUC,sensitivity,specificity,precision,NPV
0,7,0.949176,0.710169,0.960960,0.900239,0.896327
1,2,0.946855,0.735356,0.941541,0.868469,0.887102
2,9,0.943016,0.460464,0.987015,0.948749,0.809420
3,8,0.938256,0.194328,0.995925,0.947849,0.731171
4,10,0.936032,0.841556,0.883521,0.785394,0.925551
5,3,0.932420,0.610455,0.974687,0.906987,0.842404
6,5,0.931741,0.707620,0.937661,0.883706,0.878696
7,1,0.929208,0.561018,0.819406,0.809404,0.855110
8,6,0.925650,0.535587,0.971307,0.916803,0.831483
9,4,0.917029,0.506740,0.941174,0.861231,0.824142



===== Test Set Metrics for Best Fold (Fold 7) =====
AUC: 0.9710 | Sensitivity: 0.7655 | Specificity: 0.9682 | Precision: 0.9290 | NPV: 0.9155

Per-drug metrics on test set:


,drug,AUC,sensitivity,specificity,precision,NPV
0,RIF_BINARY_PHENOTYPE,0.968285,0.782946,0.977626,0.946136,0.899731
1,INH_BINARY_PHENOTYPE,0.978261,0.947210,0.942111,0.924890,0.959538
2,EMB_BINARY_PHENOTYPE,0.966324,0.566474,0.984975,0.915888,0.887218


In [5]:


#print the above test results in a table similar to the per-drug metrics table
test_summary_df = pd.DataFrame({
    "drug": ["INH_BINARY_PHENOTYPE", "RIF_BINARY_PHENOTYPE", "EMB_BINARY_PHENOTYPE"],
      "AUC":                  [0.9685, 0.9735, 0.9710],
    "sensitivity": [0.9270, 0.9133, 0.8902],
    "specificity":          [0.9628, 0.9586, 0.9482],
    "precision":            [0.9610, 0.9365, 0.8280],
    "NPV":                  [0.9302, 0.9429, 0.9686],
  
})
print("\n===== Test Set Summary Table =====")
display(test_summary_df)    


===== Test Set Summary Table =====


,drug,AUC,sensitivity,specificity,precision,NPV
0,INH_BINARY_PHENOTYPE,0.9685,0.9270,0.9628,0.9610,0.9302
1,RIF_BINARY_PHENOTYPE,0.9735,0.9133,0.9586,0.9365,0.9429
2,EMB_BINARY_PHENOTYPE,0.9710,0.8902,0.9482,0.8280,0.9686
